In [3]:
# ── crear_base_datos.ipynb ─────────────────────────────
# Script que genera la base de datos SQLite desde el CSV

import pandas as pd
import sqlite3

CSV_PATH = "hantavirus.csv"
DB_PATH  = "hantavirus.db"

# Cargar el CSV
df = pd.read_csv(CSV_PATH)
print(f"CSV cargado: {len(df)} registros, {len(df.columns)} columnas")

# Crear la base de datos SQLite 
conn = sqlite3.connect(DB_PATH)

df.to_sql("casos", conn, if_exists="replace", index=False)
print("Tabla 'casos' creada")

resumen = df.groupby("country").agg(
    total_casos=("confirmed_cases", "sum"),
    total_muertes=("deaths", "sum"),
    total_recuperados=("recovered", "sum"),
    promedio_letalidad=("case_fatality_rate", "mean"),
    años_registrados=("year", "nunique")
).reset_index()

resumen.to_sql("resumen_pais", conn, if_exists="replace", index=False)
print("Tabla 'resumen_pais' creada")

conn.close()
print(f" Base de datos guardada: {DB_PATH}")

CSV cargado: 939 registros, 15 columnas
Tabla 'casos' creada
Tabla 'resumen_pais' creada
 Base de datos guardada: hantavirus.db


In [6]:
# Exportar archivo SQL
import sqlite3

conn = sqlite3.connect("hantavirus.db")
lines = []

lines.append("-- Tabla principal de casos")
lines.append("DROP TABLE IF EXISTS casos;")
lines.append("CREATE TABLE casos (year INTEGER, iso3 TEXT, country TEXT, who_region TEXT, latitude REAL, longitude REAL, syndrome TEXT, confirmed_cases INTEGER, deaths INTEGER, recovered INTEGER, case_fatality_rate REAL, hospitalized INTEGER, icu_admissions INTEGER, human_to_human_cases INTEGER, mortality_rate_per_100k REAL);")
lines.append("")

lines.append("-- Datos tabla casos")
for row in conn.execute("SELECT * FROM casos").fetchall():
    lines.append(f"INSERT INTO casos VALUES {row};")

lines.append("")
lines.append("-- Tabla resumen por pais")
lines.append("DROP TABLE IF EXISTS resumen_pais;")
lines.append("CREATE TABLE resumen_pais (country TEXT, total_casos INTEGER, total_muertes INTEGER, total_recuperados INTEGER, promedio_letalidad REAL, anos_registrados INTEGER);")
lines.append("")

lines.append("-- Datos tabla resumen_pais")
for row in conn.execute("SELECT * FROM resumen_pais").fetchall():
    lines.append(f"INSERT INTO resumen_pais VALUES {row};")

conn.close()

with open("hantavirus_base_datos.sql", "w", encoding="utf-8") as f:
    f.write("\n".join(lines))